# Asset Master Profiling

## Purpose / scope

This notebook profiles the raw GE asset register (`asset register.xlsx`) --
the canonical spreadsheet source that will feed the first `core` asset
dimensions:

- `core.dim_make`
- `core.dim_model`
- `core.dim_owner`
- `core.dim_usage_type`
- `core.dim_asset_category`
- `core.dim_asset`

**This notebook is exploratory/discovery only.** It does not create database
tables, migrations, or production code, and it does not modify the source
file. The goal is to understand what the data actually contains -- coverage,
identity integrity, controlled-dimension candidates, and known conflicts --
so that dimension design can start from evidence rather than assumptions.

Cleaning here is limited to one safe, well-defined rule: fake missing-value
tokens (`unknown`, `N/A`, blank strings, etc.) are converted to real nulls
*for profiling purposes*. No other values are rewritten, merged, or
normalized -- see "Null/unknown cleanup" below for the exact rule.


## Load dataset

Source: the asset register spreadsheet, loaded read-only. Column names are
lower-cased/stripped for consistency; no values are changed at this step.


In [4]:
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

SOURCE_FILE = r"C:\Users\GETLS-Analytics\Downloads\asset register.xlsx"

df_raw = pd.read_excel(SOURCE_FILE)
df_raw.columns = [str(col).lower().strip() for col in df_raw.columns]

print(f"Loaded {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
df_raw.head()

Loaded 613 rows, 15 columns


,asset_number,partner_code,serial_number,model,sale_year,owner,warranty_valid,tracking,state,asset_category,brand,dispersal_date,stage,size,asset_type
0,1,unknown,unknown,unknown,unknown,Ganthorpe Enterprises,No,No Tracking,Company Equipment,Workshop Equipment,Unknown,NaT,Misc,0.0,Workshop Equipment
1,2,unknown,unknown,Unknown,unknown,Total Property Solutions (TPS),No,No Tracking,Service Contract,Construction/Mining,Unknown,NaT,Dumper,0.0,Dumper
2,3,unknown,unknown,Unknown,unknown,Manica Zimbabwe Ltd,No,No Tracking,Service Contract,Material Handling,Unknown,NaT,unknown,0.0,Forklift
3,6,unknown,308FG25-99432,8FG25,2023,Tobacco Processors Zim (TPZ),No,No Tracking,Service Contract,Material Handling,Toyota,NaT,2.5t / 3s,2.5,Diesel Forklift
4,7,unknown,unknown,CPCD50,unknown,Cross Country Containers,No,No Tracking,Service Contract,Material Handling,Hangcha,NaT,5t,5.0,Diesel Forklift


## Basic dataset profile

Row/column counts, dtypes, and overall null rate before any cleanup.


In [5]:
print(f"Rows: {df_raw.shape[0]}")
print(f"Columns: {df_raw.shape[1]}")
print()
print(df_raw.dtypes)

Rows: 613
Columns: 15

asset_number               int64
partner_code              object
serial_number             object
model                     object
sale_year                 object
owner                     object
warranty_valid            object
tracking                  object
state                     object
asset_category            object
brand                     object
dispersal_date    datetime64[ns]
stage                     object
size                     float64
asset_type                object
dtype: object


In [6]:
null_counts = df_raw.isna().sum().rename("null_count")
null_pct = (df_raw.isna().mean() * 100).round(1).rename("null_pct")
pd.concat([null_counts, null_pct], axis=1)

,null_count,null_pct
asset_number,0,0.0
partner_code,0,0.0
serial_number,0,0.0
model,0,0.0
sale_year,0,0.0
owner,0,0.0
warranty_valid,0,0.0
tracking,0,0.0
state,0,0.0
asset_category,0,0.0


## Null/unknown cleanup

The source uses several placeholder strings for "no value" instead of a real
blank cell: `unknown`, `Unknown`, `UNKNOWN`, `unknown owner`, `N/A`, `NA`,
`none`, `null`, blank strings, and whitespace-only strings. These are
converted to actual nulls (`NaN`) for every text column, so downstream
population/coverage counts are honest.

**This is the only automatic cleaning rule applied in this notebook.**
Nothing else is normalized, title-cased, merged, or rewritten -- real values
(including odd-looking ones like `Sales Stock` or `Case IH (via Agricon)`)
are preserved exactly as sourced for business review later.


In [7]:
FAKE_NULL_TOKENS = {"unknown", "n/a", "na", "none", "null", "nan", ""}


def clean_fake_nulls(series: pd.Series) -> pd.Series:
    """Blank out fake missing-value tokens; leave every real value untouched."""
    stripped = series.astype(str).str.strip()
    is_fake_null = stripped.str.lower().isin(FAKE_NULL_TOKENS)
    return stripped.mask(is_fake_null | series.isna())


df = df_raw.copy()
text_columns = df.select_dtypes(include="object").columns.tolist()

fake_null_counts = {}
for col in text_columns:
    before_na = df[col].isna().sum()
    df[col] = clean_fake_nulls(df[col])
    after_na = df[col].isna().sum()
    fake_null_counts[col] = after_na - before_na

print("Fake missing-value tokens converted to null, by column:")
pd.Series(fake_null_counts, name="converted_to_null").sort_values(ascending=False)

Fake missing-value tokens converted to null, by column:


model             131
brand              99
serial_number      79
sale_year          76
partner_code       30
stage              19
asset_type          5
asset_category      3
state               0
tracking            0
warranty_valid      0
owner               0
Name: converted_to_null, dtype: int64

In [8]:
print(f"Dataset still contains {df.shape[0]} rows and {df.shape[1]} columns")
print("(cleanup only converts values to null -- it never removes rows or columns)")

Dataset still contains 613 rows and 15 columns
(cleanup only converts values to null -- it never removes rows or columns)


## Asset identity validation

`asset_number` is intended to be the unique GE fleet/asset identifier.


In [9]:
dup_asset_mask = df["asset_number"].duplicated(keep=False)

print("Rows:", len(df))
print("Populated asset_number:", df["asset_number"].notna().sum())
print("Unique asset_number:", df["asset_number"].nunique(dropna=True))
print("Missing asset_number:", df["asset_number"].isna().sum())
print("Duplicate asset_number rows:", dup_asset_mask.sum())

Rows: 613
Populated asset_number: 613
Unique asset_number: 613
Missing asset_number: 0
Duplicate asset_number rows: 0


In [10]:
if dup_asset_mask.any():
    display(df.loc[dup_asset_mask].sort_values("asset_number"))
else:
    print("No duplicate asset_number values -- asset_number is a clean unique key.")

No duplicate asset_number values -- asset_number is a clean unique key.


## Serial-number validation

Duplicates are reported, not resolved -- a shared serial number between two
`asset_number`s is a business question (data entry error vs. genuinely
reused/reissued serial), not something this notebook should guess at.


In [11]:
serial = df["serial_number"]
dup_serial_mask = serial.notna() & serial.duplicated(keep=False)

print("Populated serial_number:", serial.notna().sum())
print("Missing serial_number:", serial.isna().sum())
print("Unique serial_number:", serial.dropna().nunique())
print("Duplicate serial_number rows:", dup_serial_mask.sum())

Populated serial_number: 534
Missing serial_number: 79
Unique serial_number: 531
Duplicate serial_number rows: 6


In [12]:
if dup_serial_mask.any():
    display(
        df.loc[dup_serial_mask, ["asset_number", "serial_number", "brand", "model", "owner"]]
        .sort_values("serial_number")
    )
else:
    print("No duplicate serial_number values.")

,asset_number,serial_number,brand,model,owner
74,450,027FDK40,Toyota,7FDK45,Full Source
75,458,027FDK40,Toyota,7FDK40,Sales Stock
107,982,2982,Field King,NaN,Ganthorpe Enterprises
155,1982,2982,NaN,NaN,Ganthorpe Enterprises
94,819,AGB 0819,Toyota,Coaster,Groundup Investments
149,1819,AGB 0819,Toyota,Corsa,Intrachem


## Controlled-dimension profiling

Candidate source fields for the first controlled dimensions:

| Source field     | Proposed dimension       |
|-------------------|--------------------------|
| `brand`            | `dim_make`               |
| `model`             | `dim_model`               |
| `owner`             | `dim_owner`                |
| `state`             | `dim_usage_type` (see note below) |
| `asset_category`    | `dim_asset_category`       |

`state` does not hold lifecycle states in the usual sense -- its actual
values (`attachment`, `company equipment`, `hire`, `sale`, `scrap`,
`service contract`, `sub contract`) describe how an asset is currently being
used/held commercially. It is profiled here as the source field for the
proposed `usage_type` dimension, not renamed in the data itself.


In [13]:
CONTROLLED_DIM_COLUMNS = ["brand", "model", "owner", "state", "asset_category"]

for col in CONTROLLED_DIM_COLUMNS:
    values = df[col].dropna()
    print(f"--- {col} ---")
    print("Populated:", values.shape[0])
    print("Null:", df[col].isna().sum())
    print("Distinct:", values.nunique())
    print()

--- brand ---
Populated: 514
Null: 99
Distinct: 48

--- model ---
Populated: 482
Null: 131
Distinct: 146

--- owner ---
Populated: 613
Null: 0
Distinct: 104

--- state ---
Populated: 613
Null: 0
Distinct: 7

--- asset_category ---
Populated: 610
Null: 3
Distinct: 9



In [14]:
print("brand -- sorted distinct values:")
sorted(df["brand"].dropna().unique().tolist())

brand -- sorted distinct values:


['Agricon',
 'Aksa',
 'Anhui Yufeng',
 'Bain',
 'Bowie',
 'CBE',
 'Case',
 'Cyclone-Bowie',
 'DFLIFT',
 'Doosan',
 'Eicher',
 'Farmtrac',
 'Field King',
 'Freightliner',
 'GEHL',
 'Generic',
 'Hangcha',
 'Heli',
 'Henred',
 'Hino',
 'Husqvarna',
 'Hyster',
 'IMining',
 'Ingersoll Rand',
 'Isuzu',
 'Iveco',
 'JAC',
 'Karcher',
 'Kipor',
 'LTMG',
 'MAN',
 'Manhand',
 'Manilec',
 'Manitou',
 'Massey Ferguson',
 'Mitsubishi',
 'Morrison',
 'Nichiyu',
 'Nissan',
 'Nissan Diesel',
 'Pamodzi',
 'Powerstar',
 'Shacman',
 'Suzuki',
 'TCM',
 'Toyota',
 'UD Trucks',
 'Yeoshe']

In [15]:
print("model -- sorted distinct values:")
sorted(df["model"].dropna().unique().tolist())

model -- sorted distinct values:


['02-8FD25',
 '1.5L',
 '150 AETJ-C 3D',
 '1500R',
 '186F',
 '1m x 2m HD Cage',
 '20-8FD25',
 '40-G2A11LI / ZSM-470 / LC45',
 '52-8FD25',
 '570T',
 '5FD70',
 '72-5FD30',
 '7FB25',
 '7FBE18',
 '7FBH25',
 '7FDA40',
 '7FDK40',
 '7FDK45',
 '8.5T GVW',
 '8FBN25',
 '8FD25',
 '8FD35',
 '8FG25',
 'AL140',
 'APD 35A',
 'Axio',
 'Bucket',
 'CAN.J 1939',
 'CBD 20J-LI3',
 'CBD20-150',
 'CBD20J',
 'CBR900',
 'CDD16J-K-LI',
 'COMPRY23',
 'CPCD25',
 'CPCD25-M3H',
 'CPCD25-Q25K2',
 'CPCD25-WSIK2',
 'CPCD30',
 'CPCD30-W11Y2G3',
 'CPCD35',
 'CPCD40-QC13K2',
 'CPCD50',
 'CPCD50-RXW35',
 'CPCD70-RW14',
 'CPD18',
 'CPD25',
 'CPD25-GB2LI',
 'CPD25-GB2LI-M',
 'CPD25-GB2LI-S',
 'CPD30-GB2LI',
 'CPD30-GB2LI-S',
 'CPD38-GB2LI-M3',
 'CPD38-GB2LI-S',
 'CPD38-ZSM470',
 'CPD45-G2A11LI',
 'CPD45-GB2LI',
 'CPD45-ZSM470',
 'CQD14-GB2SLI',
 'CQD16-GB2SLI',
 'CQD20-GB2SHDLI',
 'CQDE20C-45SZ',
 'CTD16-360',
 'Coaster',
 'Columbia',
 'Compv30',
 'Condour',
 'Corsa',
 'D25S-5',
 'Dutro',
 'EP 8 S',
 'EPT20-15',
 'EPT20-20RA

In [16]:
print("owner -- sorted distinct values:")
sorted(df["owner"].dropna().unique().tolist())

owner -- sorted distinct values:


['AFGRI Equipment Zimbabwe',
 'Abbiamo Trading (Pvt) Ltd',
 'Access Finance (Pvt) Ltd',
 'Acol Chemical Holdings',
 'Agricon Equipment Zim',
 'All Solar Zim P/L',
 'Amopra Investments',
 'Aqua Crystal Pvt Ltd',
 'Baletta Farm',
 'Blovetac Construction (Pvt) Ltd',
 'Blue Skies Zimbabwe',
 'Brandwire Distribution',
 'Brown Engineering Group (Pvt) Ltd',
 'Bullred Distribution (BRD)',
 'CEVA Logistics Zimbabwe',
 'Case IH (via Agricon)',
 'Century One Consultants',
 'ClassEcon Roofing Africa',
 'Consignment Stock',
 'Consolidated Tobacco (CTP)',
 'Crest Poultry Group',
 'Cross Country Containers',
 'Curverid Tobacco P/L',
 'CutRag Processors',
 'Dallaglio / Delta Gold',
 'Drip-tech Irrigation',
 'FX Logistics',
 'Farmec',
 'Fliknik Enterprises (Pvt) Ltd',
 'Flynote Investments',
 'Freda Rebecca Gold Mine',
 'Full Source',
 'Ganthorpe Enterprises',
 'Gold Star Sugar',
 "Golly's Wholesale",
 'Groundup Investments',
 'Gulfzone',
 'Guphil Logistics',
 'Histride Investments P/L',
 'HugeQuip / P

In [17]:
print("state -- sorted distinct values (proposed usage_type):")
sorted(df["state"].dropna().unique().tolist())

state -- sorted distinct values (proposed usage_type):


['Attachment',
 'Company Equipment',
 'Hire',
 'Sale',
 'Scrap',
 'Service Contract',
 'Sub Contract']

In [18]:
print("asset_category -- sorted distinct values:")
sorted(df["asset_category"].dropna().unique().tolist())

asset_category -- sorted distinct values:


['Access Platforms',
 'Agriculture',
 'Attachments',
 'Construction/Mining',
 'General Plant',
 'Material Handling',
 'Mining',
 'Transport',
 'Workshop Equipment']

## Make/model relationship validation

Checks whether `model` uniquely implies a `brand` (make). A model string
appearing under more than one brand is reported as a conflict for business
review -- it is **not** assumed to be the same canonical model just because
the text matches (e.g. generic capacity-based codes like `CPCD30` are
plausibly reused across manufacturers).


In [19]:
brand_model = (
    df[["brand", "model"]]
    .dropna(subset=["model"])
    .drop_duplicates()
    .sort_values(["model", "brand"])
)

model_brand_counts = brand_model.groupby("model")["brand"].nunique(dropna=True)
multi_brand_models = model_brand_counts[model_brand_counts > 1].sort_values(ascending=False)

print("Unique (brand, model) pairs:", len(brand_model))
print("Models appearing under multiple brands:", len(multi_brand_models))
print()
print(
    "Model present, brand missing:",
    ((df["model"].notna()) & (df["brand"].isna())).sum(),
)
print(
    "Brand present, model missing:",
    ((df["brand"].notna()) & (df["model"].isna())).sum(),
)
print(
    "Both populated:",
    ((df["brand"].notna()) & (df["model"].notna())).sum(),
)

Unique (brand, model) pairs: 153
Models appearing under multiple brands: 2

Model present, brand missing: 31
Brand present, model missing: 63
Both populated: 451


In [20]:
print("Model/brand conflicts -- every row for a model seen under more than one brand:")
display(
    brand_model[brand_model["model"].isin(multi_brand_models.index)]
    .sort_values(["model", "brand"])
)

Model/brand conflicts -- every row for a model seen under more than one brand:


,brand,model
578,Heli,CPCD30
139,JAC,CPCD30
201,Heli,CPD30-GB2LI
457,Hino,CPD30-GB2LI
442,Manitou,CPD30-GB2LI
478,Massey Ferguson,CPD30-GB2LI
458,Suzuki,CPD30-GB2LI
431,Toyota,CPD30-GB2LI
432,NaN,CPD30-GB2LI


## Other suspicious-field profiling

### Owner review

Flags owner values that may not represent a normal owner *organization* --
consignment/sales-stock placeholders, `/`-joined names, and `via`-brokered
names. **Reported only, not changed** -- business must confirm which of
these are legitimate values (e.g. a `P/L` company-type suffix is a normal
legal form, not a data problem) versus placeholders that should eventually
map to a real owner or an explicit "no owner" concept.


In [21]:
owner = df["owner"].dropna()

suspicious_owner_mask = (
    owner.str.lower().str.contains("sales stock")
    | owner.str.lower().str.contains("consignment stock")
    | owner.str.contains("/", regex=False)
    | owner.str.lower().str.contains(r"\bvia\b", regex=True)
)

print("Asset rows with a flagged owner value:", suspicious_owner_mask.sum())
print("Distinct owner values flagged:", owner[suspicious_owner_mask].nunique())
print()
print("Distinct flagged owner values:")
sorted(owner[suspicious_owner_mask].unique().tolist())

Asset rows with a flagged owner value: 43
Distinct owner values flagged: 9

Distinct flagged owner values:


['All Solar Zim P/L',
 'Case IH (via Agricon)',
 'Consignment Stock',
 'Curverid Tobacco P/L',
 'Dallaglio / Delta Gold',
 'Histride Investments P/L',
 'HugeQuip / PlantQuip',
 'Manitou SA (via Ganthorpe)',
 'Sales Stock']

### Remaining fields

`tracking`, `stage`, `size`, `sale_year`, `dispersal_date`, `warranty_valid`
-- profiled individually below. None of these block the first asset
dimension; see the readiness report at the end.


In [22]:
print("sale_year")
sale_year_numeric = pd.to_numeric(df["sale_year"], errors="coerce")
non_numeric_year = df["sale_year"].notna() & sale_year_numeric.isna()
implausible_year = sale_year_numeric[(sale_year_numeric < 1980) | (sale_year_numeric > 2026)]

print("  Populated:", df["sale_year"].notna().sum())
print("  Missing:", df["sale_year"].isna().sum())
print("  Minimum:", sale_year_numeric.min())
print("  Maximum:", sale_year_numeric.max())
print("  Invalid / non-numeric values:", non_numeric_year.sum())
print("  Implausible values (< 1980 or > 2026):", implausible_year.shape[0])

sale_year
  Populated: 537
  Missing: 76
  Minimum: 2000.0
  Maximum: 2026.0
  Invalid / non-numeric values: 0
  Implausible values (< 1980 or > 2026): 0


In [23]:
print("dispersal_date")
dispersal = df["dispersal_date"]
# Already parsed to datetime64 by pd.read_excel; "unparseable" would only
# arise if Excel stored a non-date value in the column, which it does not.
unparseable = pd.to_datetime(dispersal, errors="coerce").isna() & dispersal.notna()

print("  Populated:", dispersal.notna().sum())
print("  Parseable:", dispersal.notna().sum() - unparseable.sum())
print("  Unparseable:", unparseable.sum())
print("  Min parsed date:", dispersal.min())
print("  Max parsed date:", dispersal.max())

dispersal_date
  Populated: 80
  Parseable: 80
  Unparseable: 0
  Min parsed date: 2023-02-28 00:00:00
  Max parsed date: 2026-07-21 00:00:00


In [24]:
print("size")
size = df["size"]
print("  Populated:", size.notna().sum())
print("  Null:", size.isna().sum())
print("  Zero:", (size == 0).sum())
print("  Distinct non-null, non-zero values:", size[(size.notna()) & (size != 0)].nunique())
print("  Range (excluding zero):", size[size > 0].min(), "-", size[size > 0].max())

size
  Populated: 611
  Null: 2
  Zero: 171
  Distinct non-null, non-zero values: 39
  Range (excluding zero): 0.2 - 9220.0


In [25]:
print("size grouped by asset_category (non-zero only) -- min/max/count:")
display(
    df.loc[df["size"] > 0]
    .groupby("asset_category")["size"]
    .agg(["count", "min", "max"])
    .sort_values("max", ascending=False)
)

print()
print("Largest 'size' values, with category/stage/model for context:")
display(
    df.loc[df["size"] > 50, ["asset_number", "asset_category", "stage", "model", "brand", "size"]]
    .sort_values("size", ascending=False)
    .head(15)
)

size grouped by asset_category (non-zero only) -- min/max/count:


,count,min,max
asset_category,,,
Mining,2,9220.00,9220.0
Construction/Mining,93,130.00,5640.0
Transport,2,4.00,300.0
General Plant,4,270.00,270.0
Agriculture,4,5.00,90.0
Material Handling,331,0.75,7.0
Attachments,2,3.00,3.0
Access Platforms,2,0.20,3.0



Largest 'size' values, with category/stage/model for context:


,asset_number,asset_category,stage,model,brand,size
233,3671,Mining,Underground truck,MV-U40D,CBE,9220.0
486,8064,Mining,Underground truck,MV-U40D,CBE,9220.0
312,4937,Construction/Mining,Skid,Gehl 5640,GEHL,5640.0
583,9759,Construction/Mining,Telehandler,NaN,Manitou,1840.0
142,1764,Construction/Mining,Telehandler,MXT 1740 P,Manitou,1740.0
134,1600,Construction/Mining,Telehandler,MXT 1740 P,Manitou,1740.0
91,734,Construction/Mining,Telehandler,MXT 1740 P,Manitou,1740.0
522,8597,Construction/Mining,Telehandler,NaN,Manitou,1740.0
288,4260,Construction/Mining,Telehandler,NaN,Manitou,1740.0
526,8734,Construction/Mining,Telehandler,NaN,Manitou,1740.0


`size` does **not** appear to represent one consistent unit. For
`Material Handling` (forklifts) it sits in a narrow 0.75-7.0 range that
reads as lift capacity in tonnes. For `Construction/Mining` (telehandlers,
skid loaders) it instead ranges into the hundreds/thousands and lines up
with numbers embedded directly in the `model` string (e.g. `MXT 1740 P` /
`size = 1740.0`), which reads as a capacity figure in a different unit (kg)
rather than tonnes. This notebook does not assert which unit is correct for
which category -- that is a business question -- it only reports that `size`
is evidently not one consistent measure across the register.


In [26]:
print("stage")
stage = df["stage"]
print("  Distinct values:", stage.nunique(dropna=True))
print("  Populated:", stage.notna().sum())
print()
print("  Most common values:")
display(stage.value_counts(dropna=False).head(15))

stage
  Distinct values: 79
  Populated: 592

  Most common values:


stage
2.5t / 3s        81
TLB              50
2.5t / 3s / E    46
Telehandler      38
4t / 3s          34
Clamp            32
2.5t / 2s        27
3t / 3s / E      23
NaN              21
Scrap            19
3.8t / 3s / E    16
SmartTruck       13
Pallet Jack      12
4.5t / 3s / E    11
Pick up          11
Name: count, dtype: int64

`stage`'s most common values (`2.5t / 3s`, `TLB`, `Telehandler`,
`4t / 3s`, `Scaler`, `Cable Reel Handler`, ...) read as a
description/type/configuration label -- closer to a sub-type or
specification string than a lifecycle stage (e.g. "ordered / active /
disposed"). Treated here as informational, not as evidence for a
`usage_type`-style dimension.


In [27]:
print("tracking")
display(df["tracking"].value_counts(dropna=False))
print()
print("Informational only -- tracking provider will be modeled separately")
print("from physical asset identity (see asset_telemetry_assignment design).")

tracking


tracking
No Tracking    377
Sendem         141
Manitou         82
Ezy Track       13
Name: count, dtype: int64


Informational only -- tracking provider will be modeled separately
from physical asset identity (see asset_telemetry_assignment design).


In [28]:
print("warranty_valid")
display(df["warranty_valid"].value_counts(dropna=False))

warranty_valid


warranty_valid
No     456
Yes    157
Name: count, dtype: int64

## Final findings / readiness report

Summary statistics computed from the results above, followed by the
readiness assessment.


In [29]:
rows = len(df)
unique_assets = df["asset_number"].nunique(dropna=True)
dup_asset_numbers = int(dup_asset_mask.sum())

serial_populated = int(serial.notna().sum())
serial_coverage_pct = round(serial_populated / rows * 100, 1)
dup_serial_count = int(dup_serial_mask.sum())

n_makes = df["brand"].nunique(dropna=True)
n_models = df["model"].nunique(dropna=True)
n_owners = df["owner"].nunique(dropna=True)
n_usage_types = df["state"].nunique(dropna=True)
n_categories = df["asset_category"].nunique(dropna=True)
n_model_conflicts = len(multi_brand_models)
n_owner_flag_rows = int(suspicious_owner_mask.sum())
n_owner_flag_values = int(owner[suspicious_owner_mask].nunique())
n_invalid_years = int(non_numeric_year.sum()) + int(implausible_year.shape[0])
n_invalid_dispersal = int(unparseable.sum())

print("ASSET MASTER PROFILING SUMMARY")
print("=" * 40)
print(f"Rows:                        {rows}")
print(f"Unique assets:               {unique_assets}")
print(f"Duplicate asset numbers:     {dup_asset_numbers}")
print(f"Serial coverage:             {serial_populated}/{rows} ({serial_coverage_pct}%)")
print(f"Duplicate serials:           {dup_serial_count}")
print(f"Makes (brand):                {n_makes}")
print(f"Models:                       {n_models}")
print(f"Owners:                       {n_owners}")
print(f"Usage types (state):          {n_usage_types}")
print(f"Asset categories:             {n_categories}")
print(f"Make/model conflicts:         {n_model_conflicts}")
print(f"Unresolved owner values:      {n_owner_flag_rows} rows ({n_owner_flag_values} distinct)")
print(f"Invalid/implausible years:    {n_invalid_years}")
print(f"Unparseable dispersal dates:  {n_invalid_dispersal}")

ASSET MASTER PROFILING SUMMARY
Rows:                        613
Unique assets:               613
Duplicate asset numbers:     0
Serial coverage:             534/613 (87.1%)
Duplicate serials:           6
Makes (brand):                48
Models:                       146
Owners:                       104
Usage types (state):          7
Asset categories:             9
Make/model conflicts:         2
Unresolved owner values:      43 rows (9 distinct)
Invalid/implausible years:    0
Unparseable dispersal dates:  0


# Asset Master Profiling Summary

**Rows:** 613 &nbsp;|&nbsp; **Unique assets:** 613 &nbsp;|&nbsp;
**Duplicate asset numbers:** 0

**Serial coverage:** 534/613 (87.1%) populated, 531 unique &nbsp;|&nbsp;
**Duplicate serials:** 6 rows across 3 serial numbers

**Makes:** 48 &nbsp;|&nbsp; **Models:** 146 &nbsp;|&nbsp; **Owners:** 104
&nbsp;|&nbsp; **Usage types (state):** 7 &nbsp;|&nbsp; **Asset categories:** 9
(+3 null)

**Make/model conflicts:** 2 model strings shared across brands (`CPCD30`:
Heli/JAC; `CPD30-GB2LI`: Heli/Hino/Manitou/Massey Ferguson/Suzuki/Toyota)

**Unresolved owner values:** 9 distinct owner values (43 asset rows) flagged
for business review (consignment/sales-stock placeholders, `/`-joined or
`via`-brokered names -- includes ordinary `P/L` company suffixes caught by
the same `/` check)

**Invalid/implausible years:** 0 &nbsp;|&nbsp; **Unparseable dispersal
dates:** 0

**Other material quality issues:**

- `size` is not one consistent unit across `asset_category` -- small
  tonnage-range values for Material Handling vs. large model-linked values
  for Construction/Mining. Meaning not invented here; deferred to business
  review.
- `stage` behaves like a type/configuration descriptor (80 distinct values),
  not a lifecycle stage.
- `dispersal_date` populated for only 80/613 rows (13.1%) -- expected, since
  most assets are still in service.

---

### Safe to model now

- **`asset_number`** -- fully populated, fully unique. Reliable natural key
  for `core.dim_asset`.
- **`brand` -> `dim_make`** -- clean candidate; only "unknown"-token noise,
  already nulled.
- **`asset_category` -> `dim_asset_category`** -- 9 clean values (+3 null),
  no conflicts found.
- **`state` -> `dim_usage_type`** -- 7 clean commercial-usage values, no
  conflicts found.
- **`serial_number`** -- usable as a fallback/validation identifier (87.1%
  coverage); the 3 duplicate pairs are small enough for direct business
  review before modeling, not a blocker.

### Requires business review

- **The 2 make/model conflicts** (`CPCD30`, `CPD30-GB2LI` shared across
  brands) -- must be confirmed as genuinely-shared generic codes vs. a data
  entry issue before `dim_model` treats `model` as brand-scoped or global.
- **The 9 flagged owner values** -- confirm which represent real ownership
  states (e.g. `Consignment Stock`, `Sales Stock`) that `dim_owner` needs to
  model explicitly, versus which are just legal-suffix false positives
  (`P/L`) needing no action.
- **The 3 duplicate serial-number pairs** -- 2 look like distinct assets
  sharing a mistyped/reused serial (different models/owners); 1
  (`serial_number = 2982`) has almost no other descriptive data on either
  row and should be checked first.

### Deferred

- **`tracking`** -- informational; provider will be modeled separately from
  physical asset identity (`asset_telemetry_assignment`), not as a
  `dim_asset` attribute.
- **`size`** -- meaning is category-dependent and unconfirmed; do not encode
  in the first `dim_asset` until business defines the unit(s).
- **`stage`** -- reads as a type/configuration string, not a lifecycle
  state; revisit once its intended use is clarified.
- **`sale_year`**, **`dispersal_date`**, **`warranty_valid`** -- clean and
  well-behaved, but not identity/controlled-dimension fields; carry them
  into `dim_asset` as descriptive attributes without further discovery work.


# Random EDA

In [30]:
duplicated_assets = df["asset_number"][
    (~df["serial_number"].isna()) &
    (df["serial_number"].duplicated(keep=False))]
df[df["asset_number"].isin(duplicated_assets)]

,asset_number,partner_code,serial_number,model,sale_year,owner,warranty_valid,tracking,state,asset_category,brand,dispersal_date,stage,size,asset_type
74,450,G0450,027FDK40,7FDK45,2002,Full Source,No,No Tracking,Service Contract,Material Handling,Toyota,2024-03-30,4.5t / 3s,4.0,Diesel Forklift
75,458,G0458,027FDK40,7FDK40,2002,Sales Stock,No,No Tracking,Sale,Material Handling,Toyota,2024-03-30,4t / 3s,4.0,Diesel Forklift
94,819,ZIN19,AGB 0819,Coaster,2022,Groundup Investments,No,Sendem,Service Contract,Transport,Toyota,NaT,Omnibus,0.0,Bus
107,982,I0982,2982,NaN,2020,Ganthorpe Enterprises,No,No Tracking,Hire,Agriculture,Field King,NaT,Trailer,5.0,Agricultural Trailer
149,1819,NaN,AGB 0819,Corsa,NaN,Intrachem,No,No Tracking,Company Equipment,Transport,Toyota,NaT,NaN,0.0,Coaster
155,1982,I1982,2982,NaN,2018,Ganthorpe Enterprises,No,No Tracking,Hire,Agriculture,NaN,NaT,Trailer,0.0,Agricultural Trailer


In [31]:
df.loc[74, ['asset_number', 'partner_code']]

asset_number      450
partner_code    G0450
Name: 74, dtype: object

In [32]:
'''
    1982
    yif2-002982
    29825T
'''


'\n    1982\n    yif2-002982\n    29825T\n'

In [33]:
df

,asset_number,partner_code,serial_number,model,sale_year,owner,warranty_valid,tracking,state,asset_category,brand,dispersal_date,stage,size,asset_type
0,1,NaN,NaN,NaN,NaN,Ganthorpe Enterprises,No,No Tracking,Company Equipment,Workshop Equipment,NaN,NaT,Misc,0.0,Workshop Equipment
1,2,NaN,NaN,NaN,NaN,Total Property Solutions (TPS),No,No Tracking,Service Contract,Construction/Mining,NaN,NaT,Dumper,0.0,Dumper
2,3,NaN,NaN,NaN,NaN,Manica Zimbabwe Ltd,No,No Tracking,Service Contract,Material Handling,NaN,NaT,NaN,0.0,Forklift
3,6,NaN,308FG25-99432,8FG25,2023,Tobacco Processors Zim (TPZ),No,No Tracking,Service Contract,Material Handling,Toyota,NaT,2.5t / 3s,2.5,Diesel Forklift
4,7,NaN,NaN,CPCD50,NaN,Cross Country Containers,No,No Tracking,Service Contract,Material Handling,Hangcha,NaT,5t,5.0,Diesel Forklift
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
608,13840,D3840,05025DT3840,CPD25-GB2LI-S,2026,Ganthorpe Enterprises,Yes,No Tracking,Hire,Material Handling,Heli,NaT,2.5t / 3s / E,2.5,Electric Forklift (Li-Ion)
609,13841,D3841,05025DT3841,CPD25-GB2LI-S,2026,Ganthorpe Enterprises,Yes,No Tracking,Hire,Material Handling,Heli,NaT,2.5t / 3s / E,2.5,Electric Forklift (Li-Ion)
610,13845,D??45,05025DT3845,CPD25-GB2LI-S,2026,Consignment Stock,Yes,No Tracking,Sale,Material Handling,Heli,NaT,2.5t / 3s / E,2.5,Electric Forklift (Li-Ion)
611,13846,D??46,05025DT3846,CPD25-GB2LI-S,2026,Consignment Stock,Yes,No Tracking,Sale,Material Handling,Heli,2026-04-27,2.5t / 3s / E,2.5,Electric Forklift (Li-Ion)


In [34]:
df.state.value_counts()

state
Service Contract     308
Hire                 165
Attachment            45
Company Equipment     35
Scrap                 26
Sale                  24
Sub Contract          10
Name: count, dtype: int64

In [35]:
df.asset_category.value_counts()

asset_category
Material Handling      351
Construction/Mining     99
Attachments             52
Transport               46
General Plant           31
Mining                  15
Agriculture             10
Access Platforms         5
Workshop Equipment       1
Name: count, dtype: int64

In [36]:
df.brand.value_counts()

brand
Manitou            209
Heli               146
Toyota              39
Nissan              11
Powerstar           10
Mitsubishi           9
GEHL                 9
Henred               7
Case                 6
Manhand              5
Ingersoll Rand       4
JAC                  4
DFLIFT               4
TCM                  3
Suzuki               3
Iveco                3
Shacman              2
CBE                  2
Hino                 2
Morrison             2
Hangcha              2
Bowie                2
LTMG                 2
Hyster               2
UD Trucks            2
MAN                  2
Bain                 1
Yeoshe               1
Nissan Diesel        1
Field King           1
Generic              1
Karcher              1
Kipor                1
Doosan               1
Cyclone-Bowie        1
Nichiyu              1
IMining              1
Isuzu                1
Freightliner         1
Farmtrac             1
Anhui Yufeng         1
Manilec              1
Massey Ferguson      1
Agric

In [37]:
df.model.value_counts()

model
CPD30-GB2LI                    88
MI 25 D                        69
MBL-X 920                      42
MI-X 40 D                      21
Manlift 40 D                   11
MI 35 D                         7
CPD45-G2A11LI                   7
FD25                            7
CPCD25-Q25K2                    6
EPT20-15                        5
Tanker trailer                  5
NP200                           5
CPD25-GB2LI-S                   5
CPCD25                          4
MXT 1740 P                      4
TLB 818S                        4
CPD45-ZSM470                    4
FT Series                       4
CQD16-GB2SLI                    3
Hilux                           3
CPCD30                          3
Hilux Revo                      3
MX 50                           3
40-G2A11LI / ZSM-470 / LC45     3
CPCD35                          3
CPCD40-QC13K2                   3
52-8FD25                        3
CBD20-150                       3
Y1F2A25U                        3
PSE7.5B-

In [38]:
df['size'].value_counts()

size
0.00       171
2.50       169
4.00        58
3.00        48
920.00      44
3.80        16
840.00      13
1740.00      8
2.00         7
3.50         7
1.00         7
1440.00      6
4.50         6
5.00         5
1.80         4
270.00       4
818.00       4
7.00         4
1.60         3
732.00       3
300.00       2
9220.00      2
75.00        2
733.00       2
1.50         1
0.20         1
570.00       1
1240.00      1
1340.00      1
0.75         1
90.00        1
1500.00      1
130.00       1
5640.00      1
625.00       1
735.00       1
790.00       1
742.00       1
634.00       1
1840.00      1
Name: count, dtype: int64

# Asset Type

In [39]:
print(f"No. of asset types before normalizations: {df.asset_type.nunique()}")
df["asset_type"].value_counts()

No. of asset types before normalizations: 95


asset_type
Diesel Forklift                  171
Electric Forklift (Li-Ion)       103
Backhoe Loader (TLB)              49
Telehandler                       30
Diesel Manlift                    18
Tobacco Clamp                     16
Attachment                        13
Electric Pallet Truck             11
Electric Forklift                 11
Forklift                           9
Skid Steer Loader                  8
Pickup Truck (Bakkie)              8
Charger/Service Truck              8
Transfer Pump                      7
Pickup Truck                       7
Rigid Truck                        6
Rough Terrain Forklift             6
Tanker Trailer                     5
Electric Reach Truck               5
Charging/Service Truck             5
Diesel Forklift (Scrap)            4
Utility Trailer                    3
Agricultural Tractor               3
Agricultural Trailer               3
Reciprocating Compressor           3
Personnel Work Platform            3
Air Compressor             

In [40]:
CANONICAL_ASSET_TYPE_MAP = {
    # Forklifts
    "forklift": "Forklift",
    "diesel forklift": "Diesel Forklift",
    "diesel forklift (scrapped)": "Diesel Forklift",
    "diesel forklift (scrap)": "Diesel Forklift",
    "diesel forklift (closed cab)": "Diesel Forklift",
    "diesel forklift (triplex)": "Diesel Forklift",
    "electric forklift": "Electric Forklift",
    "electric forklift (li-ion)": "Electric Forklift",
    "chinese electric forklift": "Electric Forklift",
    "lpg forklift": "LPG Forklift",
    "petrol forklift": "Petrol Forklift",
    "rough terrain forklift": "Diesel Forklift",

    # Warehouse / pallet equipment
    "electric pallet truck": "Pallet Truck",
    "electric pallet jack": "Pallet Truck",
    "electric ride-on pallet truck": "Pallet Truck",
    "electric pallet stacker": "Pallet Stacker",
    "electric stacker": "Pallet Stacker",
    "straddle electric stacker": "Pallet Stacker",
    "electric reach truck": "Reach Truck",
    "electric reach truck (li-ion)": "Reach Truck",
    "hd electric reach truck": "Reach Truck",

    # Material handling
    "telehandler": "Telehandler",
    "telehandler (anglo spec)": "Telehandler",
    "telehandler (mining spec)": "Telehandler",
    "telehandler (agri spec)": "Telehandler",
    "telehandler (jd engine)": "Telehandler",
    "compact telehandler": "Telehandler",
    "heavy telehandler": "Telehandler",
    "mining telehandler": "Telehandler",

    # Earthmoving
    "dumper": "Dumper",
    "loader bucket": "Attachment",
    "wheel loader": "Loader",
    "skid steer loader": "Loader",
    "backhoe loader (tlb)": "Backhoe Loader",

    # Agricultural
    "agricultural tractor": "Tractor",
    "4wd tractor": "Tractor",
    "straddle tractor": "Tractor",

    # Trucks
    "rigid truck": "Truck",
    "box truck": "Truck",
    "charger/service truck": "Service Truck",
    "charging/service truck": "Service Truck",
    "mechanical horse": "Truck",

    # Light vehicles
    "pickup truck": "Pickup",
    "pickup truck (bakkie)": "Pickup",
    "utility vehicle": "Utility Vehicle",
    "light utility vehicle": "Utility Vehicle",
    "light utility van/truck": "Utility Vehicle",
    "passenger car": "Passenger Vehicle",
    "passenger car (sedan)": "Passenger Vehicle",
    "bus": "Bus",
    "coaster": "Bus",

    # Trailers
    "trailer": "Trailer",
    "utility trailer": "Trailer",
    "agricultural trailer": "Agricultural Trailer",
    "farm trailer": "Agricultural Trailer",
    "tanker trailer": "Tanker Trailer",
    "tanker trailer (front)": "Tanker Trailer",
    "tanker trailer (rear)": "Tanker Trailer"
    "dumper trailer", "Dumper Trailer"
    "water trailer": "Water Trailer",
    # Compressors
    "reciprocating compressor": "Compressor",
    "electric air compressor": "Compressor",
    "air compressor": "Compressor",
    "rotary screw compressor": "Compressor",

    # Pumps
    "pump": "Pump",
    "water pump": "Pump",
    "transfer pump": "Pump",
    "rotary gear pump": "Pump",

    # Power equipment
    "diesel generator": "Generator",
    "diesel engine": "Engine",
    "hydraulic power pack": "Hydraulic Power Pack",

    # Access / lifting
    "personnel work platform": "Access Platform",
    "access platform": "Access Platform",
    "hydraulic scissor lift": "Scissor Lift",
    "rt scissor lift": "Scissor Lift",
    "diesel manlift": "Boom Lift",
    "electric boom lift": "Boom Lift",
    "towable boom lift": "Boom Lift",
    "hydraulic lifting platform": "Lifting Platform",

    # Workshop / cleaning
    "workshop equipment": "Workshop Equipment",
    "bandsaw": "Workshop Equipment",
    "high pressure cleaner": "Cleaning Equipment",
    "high pressure washer": "Cleaning Equipment",
    "concrete/surface scaler": "Surface Scaler",

    # Attachments
    "attachment": "Attachment",
    "bale clamp": "Attachment",
    "loader bucket": "Attachment",
    "fork frame/tines": "Attachment",
    "tobacco clamp": "Attachment",
    "telehandler bucket": "Attachment",
    "hydraulic fork bucket": "Attachment",
    "personnel cage": "Attachment",
    "crane jib attachment": "Attachment",
    "cable reel handler": "Attachment",
    "tyre handler": "Attachment",

    # Explicitly unresolved
    "unknown": "Other Equipment",
    "": "Other Equipment",
    "nan": "Other Equipment",
    "electric motor": "Other Equipment",
}

In [41]:
unmapped = (
    df.loc[
        ~df["asset_type"].str.strip().str.lower().isin(CANONICAL_ASSET_TYPE_MAP),
        "asset_type"
    ]
    .drop_duplicates()
    .sort_values()
)

print(unmapped.tolist())

['Dumper Trailer', 'Water Trailer', nan]


In [54]:
df["asset_type"] = (
    df["asset_type"]
    .str.strip()
    .str.lower()
    .map(CANONICAL_ASSET_TYPE_MAP)
)

df["asset_type"] = df["asset_type"].fillna("Other Equipment")
print(f"No. of asset types after normalizations: {df.asset_type.nunique()}")

No. of asset types after normalizations: 18


In [55]:
import re
temp_path = r"C:\Users\GETLS-Analytics\OneDrive - Ganthorpe Enterprises (Private) Ltd\Desktop\\"
save_path =  r"C:\Users\GETLS-Analytics\OneDrive - Ganthorpe Enterprises (Private) Ltd\Desktop\Projects\G.E\ge-data-platform\data\local\assets\\"
df.to_csv(save_path + "asset_master_bootstrap.csv", index=False, encoding="utf-8")

In [56]:
df.asset_type.info()

<class 'pandas.core.series.Series'>
RangeIndex: 613 entries, 0 to 612
Series name: asset_type
Non-Null Count  Dtype 
--------------  ----- 
613 non-null    object
dtypes: object(1)
memory usage: 4.9+ KB
